# Using ChatGPT or Gemini with Python and LangChain

In this notebook you will use ChatGPT or Gemini and LangChain to solve and learn about:

- Langchain chains
- Memory and conversation chains



## Install OpenAI and LangChain dependencies

In [ ]:
!pip install langchain==0.3.11
!pip install langchain-openai==0.2.12

# Optional: Install LangChain Google Gemini Dependency

Google Gemini API is free (till now). You can get a key [here](https://aistudio.google.com/app/apikey), just need to sign in with your google account. Gemini may not be available fully in EU.

In [ ]:
!pip install langchain-google-genai==2.0.7

## Load OpenAI API Credentials

Here we load it from a file so we don't explore the credentials on the internet by mistake

In [ ]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

In [ ]:
import yaml

with open('api_keys.yml', 'r') as file:
    api_creds = yaml.safe_load(file)

In [ ]:
api_creds.keys()

dict_keys(['OPENAI_API_KEY', 'GEMINI_API_KEY', 'NGORK_AUTH_TOKEN', 'DEEPSEEK_API_KEY', 'TOGETHER_API_KEY', 'TAVILY_API_KEY'])

In [ ]:
import os

os.environ['OPENAI_API_KEY'] = api_creds['OPENAI_API_KEY']

## Optional: Load Gemini API credentials

Run this section only if you are using Google Gemini

In [ ]:
# import os
# import yaml

# with open('gemini_key.yml', 'r') as file:
#     api_creds = yaml.safe_load(file)

os.environ["GOOGLE_API_KEY"] = api_creds['GEMINI_API_KEY']

## Load Necessary Dependencies and ChatGPT LLM

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [ ]:
model = ChatOpenAI(model_name='gpt-4o-mini', temperature=0.0)

## Optional: Load Google Gemini LLM

Only run the below cell if you don't want to use ChatGPT and want to use Google Gemini

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

gemini_model = ChatGoogleGenerativeAI(model="gemini-2.0-flash",
                                      convert_system_message_to_human=True)

## Let's look at how to create a basic chain

In [ ]:
PROMPT = "tell me a joke about {topic}"
prompt = ChatPromptTemplate.from_template(PROMPT)
chain = (
         prompt
         |
         model
)

response = chain.invoke({"topic": "bears"})
print(response.content)

Why do bears have hairy coats?

Because they look silly in sweaters!


In [ ]:
# same code with gemini
prompt = ChatPromptTemplate.from_template("tell me a joke about {topic}")
chain = (
         prompt
         |
         gemini_model
)

response = chain.invoke({"topic": "bears"})
print(response.content)

/usr/local/lib/python3.11/dist-packages/langchain_google_genai/chat_models.py:310: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


Why did the bear dissolve in water?

Because it was polar!



### Note: Replace model with gemini_model in any of the code below if you want to use Google Gemini instead of ChatGPT

In [ ]:
# can be used on multiple prompts also
topics = [{'topic': 'AI'}, {'topic': 'Statistics'}]
responses = chain.map().invoke(topics)

/usr/local/lib/python3.11/dist-packages/langchain_google_genai/chat_models.py:310: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")
/usr/local/lib/python3.11/dist-packages/langchain_google_genai/chat_models.py:310: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


In [ ]:
for response in responses:
  print(response.content)
  print('-----')
  print('\n')

Why did the AI cross the road?

To optimize the pedestrian traffic flow and gather data on human behavior patterns. Also, it was curious about what was on the other side, but didn't want to admit it was just being nosy.

-----


Why did the statistician bring a ladder to the bar?

Because he heard the drinks were on the house and he wanted to get a higher standard deviation!

-----




## Basic chains are ad-hoc - No conversation history!

In [ ]:
prompt = ChatPromptTemplate.from_template("{query}")
basic_chain = (
               prompt
               |
              model
)

In [ ]:
response = basic_chain.invoke({"query" : 'What are the first four colors of the rainbow?'})
print(response.content)

The first four colors of the rainbow are red, orange, yellow, and green.


In [ ]:
response = basic_chain.invoke({"query" : 'And the other three?'})
print(response.content) # gives a totally random response

It seems like your question might be missing some context. Could you please provide more details or clarify what you mean by "the other three"? This will help me give you a more accurate and helpful response.


## Let's learn how to add memory to build a conversation chain

In [ ]:
from langchain.memory import ConversationBufferWindowMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [ ]:
# create prompt template
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Act as a helpful AI Assistant"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)
# k=3 stores last 3 conversations between you and ChatGPT
memory = ConversationBufferWindowMemory(k=3, return_messages=True)

<ipython-input-19-d4bbd9cd6cde>:10: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(k=3, return_messages=True)


In [ ]:
memory.load_memory_variables({}) # shows the conversation history

{'history': []}

In [ ]:
from operator import itemgetter
# creates the conversation chain
chain = (
    RunnablePassthrough.assign(
        history=RunnableLambda(memory.load_memory_variables)
        |
        itemgetter("history")
    )
    |
    prompt
    |
    model
)

In [ ]:
user_input = {'input': 'What are the first four colors of a rainbow'}
response = chain.invoke(user_input)
response.content

'The first four colors of a rainbow, in order, are red, orange, yellow, and green.'

In [ ]:
memory.load_memory_variables({})

{'history': []}

In [ ]:
# remember to save your conversation to the memory
memory.save_context(user_input, {"output": response.content})
memory.load_memory_variables({}) # remembers the conversation history

{'history': [HumanMessage(content='What are the first four colors of a rainbow', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The first four colors of a rainbow, in order, are red, orange, yellow, and green.', additional_kwargs={}, response_metadata={})]}

In [ ]:
user_input = {'input': 'And the last 3?'}
response = chain.invoke(user_input) # uses history of the past conversation to give a better response
response.content

'The last three colors of a rainbow are blue, indigo, and violet. So, the full sequence of colors in a rainbow is red, orange, yellow, green, blue, indigo, and violet.'

In [ ]:
memory.save_context(user_input, {"output": response.content})
memory.load_memory_variables({})

{'history': [HumanMessage(content='What are the first four colors of a rainbow', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The first four colors of a rainbow, in order, are red, orange, yellow, and green.', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='And the last 3?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The last three colors of a rainbow are blue, indigo, and violet. So, the full sequence of colors in a rainbow is red, orange, yellow, green, blue, indigo, and violet.', additional_kwargs={}, response_metadata={})]}